**Document References**

1. https://docs.ultralytics.com/guides/trackzone

2. https://docs.ultralytics.com/datasets/detect/coco

3. https://github.com/ultralytics/ultralytics/blob/main/ultralytics/solutions/region_counter.py

4. https://github.com/ultralytics/ultralytics/blob/main/examples/YOLOv8-Region-Counter/yolov8_region_counter.py

# COCO Dataset Classes
source -> https://docs.ultralytics.com/datasets/detect/coco
# Classes (id -> class name ) format
0. person
1. bicycle
2. car
3. motorcycle
4. airplane
5. bus
6. train
7. truck
8. boat
9. traffic light
10. fire hydrant
11. stop sign
12. parking meter
13. bench
14. bird
15. cat
16. dog
17. horse
18. sheep
19. cow
20. elephant
21. bear
22. zebra
23. giraffe
24. backpack
25. umbrella
26. handbag
27. tie
28. suitcase
29. frisbee
30. skis
31. snowboard
32. sports ball
33. kite
34. baseball bat
35. baseball glove
36. skateboard
37. surfboard
38. tennis racket
39. bottle
40. wine glass
41. cup
42. fork
43. knife
44. spoon
45. bowl
46. banana
47. apple
48. sandwich
49. orange
50. broccoli
51. carrot
52. hot dog
53. pizza
54. donut
55. cake
56. chair
57. couch
58. potted plant
59. bed
60. dining table
61. toilet
62. tv
63. laptop
64. mouse
65. remote
66. keyboard
67. cell phone
68. microwave
69. oven
70. toaster
71. sink
72. refrigerator
73. book
74. clock
75. vase
76. scissors
77. teddy bear
78. hair drier
79. toothbrush

In [ ]:
# script to detect objects through yolo base model without zone logic
import cv2

!pip install ultralytics

from ultralytics import YOLO

cap = cv2.VideoCapture("/content/motion-detection-3.mp4")
assert cap.isOpened(), "Error reading video file"

w, h, fps = (int(cap.get(x)) for x in (cv2.CAP_PROP_FRAME_WIDTH, cv2.CAP_PROP_FRAME_HEIGHT, cv2.CAP_PROP_FPS))
video_writer = cv2.VideoWriter("/content/yolo_baseline.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

model = YOLO("yolo26n.pt")

while cap.isOpened():
    success, im0 = cap.read()
    if not success:
        print("Video frame is empty or processing is complete.")
        break

    results = model(im0,classes=[58], conf=0.4, verbose=False)[0]
    video_writer.write(results.plot())   # Ultralytics draws boxes + class + conf

cap.release()
video_writer.release()

**Results of yolo basline model (multiple classes) without zone specific detection**

1. https://res.cloudinary.com/dozk61p5d/video/upload/v1786519472/fixed_yolo_baseline_rzzrx9.mp4

2.https://res.cloudinary.com/dozk61p5d/video/upload/v1786519474/fixed_yolo_baseline_1_sz2mgb.mp4

**Results of yolo basline model (single class) without zone specific detection**

1. https://res.cloudinary.com/dozk61p5d/video/upload/v1786519475/fixed_yolo_baseline_2_ltcyu4.mp4

2. https://res.cloudinary.com/dozk61p5d/video/upload/v1786519475/fixed_yolo_baseline_3_wupgre.mp4


In [ ]:
# Original Trackzone Script
import cv2

from ultralytics import solutions

cap = cv2.VideoCapture("path/to/video.mp4")
assert cap.isOpened(), "Error reading video file"

region_points = [(150, 150), (1130, 150), (1130, 570), (150, 570)]
trackzone = solutions.TrackZone(show=False, region=region_points, model="yolo26n.pt")

while cap.isOpened():
    success, im0 = cap.read()
    if not success:
        break
    results = trackzone(im0)
    print(f"Objects currently in zone: {results.total_tracks}")  # live zone occupancy

cap.release()

In [ ]:
# modified script for object detection through trackzone with multiple zone detection
import cv2

!pip install ultralytics
from ultralytics import solutions

cap = cv2.VideoCapture("/content/Person-Cargo-walk-15.mp4")
assert cap.isOpened(), "Error reading video file"

# w, h needed before defining regions
w, h, fps = (int(cap.get(x)) for x in (cv2.CAP_PROP_FRAME_WIDTH, cv2.CAP_PROP_FRAME_HEIGHT, cv2.CAP_PROP_FPS))

# FIXED: corner points
region_points = [
    [(0, 0), (256, 0), (256, h), (0, h)],           # zone 1 — leftmost strip
    [(w - 256, 0), (w, 0), (w, h), (w - 256, h)],   # zone 2 — rightmost strip
]

# Video writer
video_writer = cv2.VideoWriter("/content/trackzone_output.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

# one TrackZone instance per zone
trackzones = [
    solutions.TrackZone(
        show=False,
        classes=[0],
        region=zone,
        model="yolo26n.pt",
        # line_width=2,
    )
    for zone in region_points
]

# Process video
while cap.isOpened():
    success, im0 = cap.read()
    if not success:
        print("Video frame is empty or processing is complete.")
        break

    results = im0
    for tz in trackzones:
        results = tz(results).plot_im

    video_writer.write(results)

cap.release()
video_writer.release()

**Results of yolo basline model (multiple classes) with zone specific detection**

1. https://res.cloudinary.com/dozk61p5d/video/upload/v1786521753/fixed_trackzone_output_15_skorne.mp4



**Results of yolo basline model (single class) with zone specific detection**

1. https://res.cloudinary.com/dozk61p5d/video/upload/v1786521752/fixed_trackzone_output_16_cevsdb.mp4



In [ ]:
# multi-zone masking with yolo, single inference pass, per-detection zone logging
import cv2
import numpy as np
!pip install ultralytics
from ultralytics import YOLO

cap = cv2.VideoCapture("/content/Person-Cargo-walk-15.mp4")
assert cap.isOpened(), "Error reading video file"

w, h, fps = (int(cap.get(x)) for x in (cv2.CAP_PROP_FRAME_WIDTH,
                                       cv2.CAP_PROP_FRAME_HEIGHT,
                                       cv2.CAP_PROP_FPS))

BORDER = 2
ZONES = [                       # name, x1, y1, x2, y2
    ("left zone",  0, 0, 256, h),
    ("right zone", w - 256, 0, w, h),
]

model = YOLO("yolo26n.pt")
video_writer = cv2.VideoWriter("/content/masked_output.mp4",
                               cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

frame_idx = 0

while cap.isOpened():
    success, im0 = cap.read()
    if not success:
        print("Video frame is empty or processing is complete.")
        break

    frame_idx += 1

    masked = np.zeros_like(im0)
    for _, x1, y1, x2, y2 in ZONES:
        masked[y1:y2, x1:x2] = im0[y1:y2, x1:x2]

    r = model(masked, classes=[0], conf=0.35, verbose=False)[0]

    # ---------- logs for zone based detection
    for bx1, by1, bx2, by2 in r.boxes.xyxy.cpu().numpy():
        cx, cy = (bx1 + bx2) / 2, (by1 + by2) / 2
        for n, x1, y1, x2, y2 in ZONES:
            if x1 <= cx < x2 and y1 <= cy < y2:
                print("%d: %s" % (frame_idx, n))
                break
    # ----------------------------------------------------

    out = r.plot(img=im0.copy())
    for _, x1, y1, x2, y2 in ZONES:
        out[y1:y1 + BORDER, x1:x2] = out[y2 - BORDER:y2, x1:x2] = 255
        out[y1:y2, x1:x1 + BORDER] = out[y1:y2, x2 - BORDER:x2] = 255

    video_writer.write(out)

cap.release()
video_writer.release()
print("written to /content/masked_output.mp4")

**Results with Custom Logic of Yolo with Zone Specific**

1. https://res.cloudinary.com/dozk61p5d/video/upload/v1786703929/fixed_masked_output_7_c4uddo.mp4

In [ ]:
# official - region counter script
# source -> https://docs.ultralytics.com/guides/region-counting
import cv2

from ultralytics import solutions

cap = cv2.VideoCapture("path/to/video.mp4")
assert cap.isOpened(), "Error reading video file"

# Pass region as list
# region_points = [(20, 400), (1080, 400), (1080, 360), (20, 360)]

# Pass region as dictionary
region_points = {
    "region-01": [(50, 50), (250, 50), (250, 250), (50, 250)],
    "region-02": [(640, 640), (780, 640), (780, 720), (640, 720)],
}

# Video writer
w, h, fps = (int(cap.get(x)) for x in (cv2.CAP_PROP_FRAME_WIDTH, cv2.CAP_PROP_FRAME_HEIGHT, cv2.CAP_PROP_FPS))
video_writer = cv2.VideoWriter("region_counting.avi", cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

# Initialize region counter object
regioncounter = solutions.RegionCounter(
    show=True,  # display the frame
    region=region_points,  # pass region points
    model="yolo26n.pt",  # model for counting in regions, e.g., yolo26s.pt
)

# Process video
while cap.isOpened():
    success, im0 = cap.read()

    if not success:
        print("Video frame is empty or processing is complete.")
        break

    results = regioncounter(im0)

    # print(results)  # access the output

    video_writer.write(results.plot_im)

cap.release()
video_writer.release()
cv2.destroyAllWindows()  # destroy all opened windows

**Updated Script Logic Below with yolo specific zone based detection through masking & rectangle drawing logic (unofficial)**

**The Script logic below includes yolo specific rectangle drawing logic similar to the one present in region counter official script but includes custom masking logic to detect objects in zone area only**

In [ ]:
# approach-1 multi-zone with yolo custom zone detection & rectangle drawing logic, single inference pass, per-detection zone logging
import cv2
import numpy as np
!pip install ultralytics
from ultralytics import YOLO
from ultralytics.solutions.solutions import SolutionAnnotator
from ultralytics.utils.plotting import colors

cap = cv2.VideoCapture("/content/Person-Cargo-walk-15.mp4")
assert cap.isOpened(), "Error reading video file"

w, h, fps = (int(cap.get(x)) for x in (cv2.CAP_PROP_FRAME_WIDTH,
                                       cv2.CAP_PROP_FRAME_HEIGHT,
                                       cv2.CAP_PROP_FPS))

BORDER = 2
ZONES = [                       # name, x1, y1, x2, y2
    ("left zone",  0, 0, 256, h),
    ("right zone", w - 256, 0, w, h),
]

model = YOLO("yolo26n.pt")
video_writer = cv2.VideoWriter("/content/masked_output.mp4",
                               cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

frame_idx = 0

while cap.isOpened():
    success, im0 = cap.read()
    if not success:
        print("Video frame is empty or processing is complete.")
        break

    frame_idx += 1

    masked = np.zeros_like(im0)
    for _, x1, y1, x2, y2 in ZONES:
        masked[y1:y2, x1:x2] = im0[y1:y2, x1:x2]

    r = model(masked, classes=[0], conf=0.35, verbose=False)[0]

    # ---------- logs for zone based detection
    for bx1, by1, bx2, by2 in r.boxes.xyxy.cpu().numpy():
        cx, cy = (bx1 + bx2) / 2, (by1 + by2) / 2
        for n, x1, y1, x2, y2 in ZONES:
            if x1 <= cx < x2 and y1 <= cy < y2:
                print("%d: %s" % (frame_idx, n))
                break
    # ----------------------------------------------------

  #------------------- Drawing Logic for Rectangle Region ------------------------------------
    ann = SolutionAnnotator(r.plot(img=im0.copy()), line_width=BORDER)  # wrap plotted frame
    for i, (_, x1, y1, x2, y2) in enumerate(ZONES):
        corners = [(x1, y1), (x2, y1), (x2, y2), (x1, y2)]              # rect -> perimeter pts
        ann.draw_region(reg_pts=corners, thickness=BORDER * 2)
    out = ann.result()                                                  # annotated frame back

  # ------------------------------------------------------------------------------------------

    video_writer.write(out)

cap.release()
video_writer.release()
print("written to /content/masked_output.mp4")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 6.7 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
101: left zone
102: left zone
103: left zone
104: left zone
105: left zone
106: left zone
107: left zone
108: left zone
108: left zone
109: left zone
110: left zone
111: left zone
112: left zone
113: left zone
114: left zone
115: left zone
116: left zone
117: left zone
118: left zone
119: left zone
120: left zone
121: left zone
122: left zone
123: left zone
124: left zone
125: left zone
126: left zone
127: left zone
128: left zone
129: left zone
130:

**Results with Custom Logic of Yolo with Zone Specific Detection & Rectangle Drawing Logic**

1. https://res.cloudinary.com/dozk61p5d/video/upload/v1786712972/fixed_masked_output_9_d3q3tl.mp4



**Updated Script Logic Below with yolo specific zone based detection & polygon drawing logic (official-ultralytics)**

Source -> https://github.com/ultralytics/ultralytics/blob/main/ultralytics/solutions/region_counter.py

In [ ]:
# approach-2 multi-zone polygon detection with yolo, no masking, no tracking
import cv2
import numpy as np
!pip install ultralytics shapely
from ultralytics import YOLO
from ultralytics.solutions.solutions import SolutionAnnotator
from ultralytics.utils.plotting import colors
from shapely.geometry import Point, Polygon
from shapely.prepared import prep

cap = cv2.VideoCapture("/content/Person-Cargo-walk-15.mp4")
assert cap.isOpened(), "Error reading video file"

w, h, fps = (int(cap.get(x)) for x in (cv2.CAP_PROP_FRAME_WIDTH,
                                       cv2.CAP_PROP_FRAME_HEIGHT,
                                       cv2.CAP_PROP_FPS))

BORDER = 2
ZONES = {                                   # name -> polygon points, any shape
    "left zone":  [(120, 80), (640, 40), (880, 320), (700, 260)],
    "right zone": [(760, 620), (420, 540), (300, 780), (180, 460), (40, 300)],
}

# build once: prepped polygon for fast tests + centroid for the name label
polys = {n: prep(Polygon(pts)) for n, pts in ZONES.items()}
cents = {n: Polygon(pts).centroid for n, pts in ZONES.items()}

model = YOLO("yolo26n.pt")
video_writer = cv2.VideoWriter("/content/zones_output.mp4",
                               cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

frame_idx = 0

while cap.isOpened():
    success, im0 = cap.read()
    if not success:
        print("Video frame is empty or processing is complete.")
        break

    frame_idx += 1

    r = model(im0, classes=[0], conf=0.35, verbose=False)[0]   # full frame, no mask

    ann = SolutionAnnotator(im0.copy(), line_width=BORDER)

    # ---------- draw zones + name label at each centroid
    for i, (n, pts) in enumerate(ZONES.items()):
        ann.draw_region(reg_pts=pts, color=colors(i, True), thickness=BORDER * 2)
        cx, cy = int(cents[n].x), int(cents[n].y)
        ann.adaptive_label([cx, cy, cx, cy], label=n, color=colors(i, True),
                           txt_color=(255, 255, 255), margin=BORDER * 4, shape="rect")
    # ----------------------------------------------------

    boxes = r.boxes.xyxy.cpu().numpy()
    confs = r.boxes.conf.cpu().numpy()

    # ---------- zone based detection: log + draw only in-zone boxes
    for box, conf in zip(boxes, confs):
        bx1, by1, bx2, by2 = box
        c = Point((bx1 + bx2) / 2, (by1 + by2) / 2)            # centroid
        for i, (n, p) in enumerate(polys.items()):
            if p.contains(c):                                  # repo-exact containment
                print("%d: %s" % (frame_idx, n))
                ann.box_label(box, "person %.2f" % conf, color=colors(i, True))
                break
    # ----------------------------------------------------

    video_writer.write(ann.result())

cap.release()
video_writer.release()
print("written to /content/zones_output.mp4")

194: left zone
195: left zone
196: left zone
197: left zone
198: left zone
198: left zone
199: left zone
199: left zone
200: left zone
201: left zone
202: left zone
203: left zone
204: left zone
204: left zone
205: left zone
206: left zone
207: left zone
208: left zone
209: left zone
210: left zone
211: left zone
212: left zone
213: left zone
214: left zone
215: left zone
216: left zone
217: left zone
218: left zone
219: left zone
220: left zone
221: left zone
222: left zone
223: left zone
224: left zone
225: left zone
226: left zone
227: left zone
228: left zone
Video frame is empty or processing is complete.
written to /content/zones_output.mp4


**Results with Official Logic of Yolo for Zone Specific Detection & Drawing Polygons**

1. https://res.cloudinary.com/dozk61p5d/video/upload/v1786712970/fixed_zones_output_1_closc4.mp4

2. https://res.cloudinary.com/dozk61p5d/video/upload/v1786712970/fixed_zones_output_3_q9fsqr.mp4

3. https://res.cloudinary.com/dozk61p5d/video/upload/v1786712972/fixed_zones_output_4_ozdwlv.mp4